# Design: Hanlon (2005), Table 3 Panel B

## Assigned scope and source evidence

This attempt implements only the full-sample pre-tax earnings persistence regression in Equation (2), reported in Table 3 Panel B. The source equation has an intercept, the LNBTD and LPBTD indicators, current PTBI, and the two current-PTBI interactions. SmallBTD is the omitted category; this is one interacted regression, not three separate group regressions. The original article's PDF physical page 9 (printed page 145) defines Equation (2), the annual within-year quintiles, and the tax-item construction; physical page 8 (printed page 144) describes the intersection sample and its screens; physical page 12 (printed page 148) states that financial statement variables are reset at the 1st and 99th percentiles and that industry controls are not used; physical page 13 (printed page 149) gives item and scaling definitions; physical page 15 (printed page 151) defines current tax for CETR as item #16 less item #50; physical page 16 (printed page 152), Table 3 Panel B, reports the N, coefficients, t-statistics, adjusted R-squared, and the F-test of equality of the two interactions.

The source reports 14,106 firm-years (4,048 firms) for this fit. This is a comparison target, not a construction rule. Table 3 reports the interaction estimates as -0.100 and -0.212, with t-statistics -4.560 and -7.990, adjusted R-squared 0.272, and an F-test of equality of 15.01 (p=0.0001). The paper states that both interaction coefficients have two-tailed p-values of 0.0001 in the results discussion (physical page 12). Industry controls, weights, and a covariance correction are not reported; the OLS table reports t-statistics and two-tailed tests. Stata `regress` therefore uses conventional model-based OLS inference, explicitly as the best-effort choice where the source is silent.

## Inputs and fields

The build reads only catalog-listed annual COMP.FUNDA, CRSP.CCMXPF_LINKTABLE, and CRSP.DSF files through read-only SAS libraries. It restricts Compustat to fiscal labels 1993–2001 to preserve the prior-year denominator and one-year lead history for the 1994–2000 analysis labels. The firm-year key is GVKEY and fiscal year; the CRSP security identifier is PERMNO. The specific Compustat fields are GVKEY, DATADATE, FYEAR, FIC, SICH, INDFMT, CONSOL, POPSRC, DATAFMT, UPD, FDATE, PDATE, CURCD, PI, MII, AT, TXC, TLCF, TXDFED, TXDFO, and TXDI. CRSP link fields are GVKEY, LINKPRIM, LINKTYPE, LPERMNO, LINKDT, and LINKENDDT; CRSP daily membership uses PERMNO and DATE only.

The field labels and crosswalk are supported by the catalog and cached provider guides indexed in `output/reproduction_runs/resource_documentation.json`. The 2004 S&P Research Insight guide maps items #6/#16/#49/#50/#52/#170/#269/#270 to AT/TXT/MII/TXDI/TLCF/PI/TXDFED/TXDFO, respectively; their physical/printed pages are recorded in the attempt-local `reference_documents/provider_field_crosswalk.md`. The corresponding current FUNDA labels match. An average-assets series is not directly present in the selected file.

## Population and construction

Compustat rows are restricted to annual industrial, consolidated, domestic-standard population records (`INDFMT=INDL`, `CONSOL=C`, `POPSRC=D`, `DATAFMT=STD`) and final updates (`UPD=3`). The article does not prescribe an update-code filter; this is a best-effort source-vintage choice based on the 2004 guide, which defines UPD=1 as not updated except for adjustment/market data, UPD=2 as preliminary, and UPD=3 as final (physical page 122; printed page 72). In fiscal labels 1993–2001, 109,696 standard rows have UPD 1, 2, or 3: 5,526 have UPD=1, 535 have UPD=2, and 103,635 have UPD=3. The filter removes 6,061 rows (5.5%), mostly UPD=1 records that lack final fundamental updates; 535 rows are preliminary. It is reasonable for a final-report data build but can exclude usable preliminary records if a final source is unavailable. The article does not settle that choice, and the source sample size is not used to choose it. For any remaining multiple records per GVKEY and fiscal-year label, construction sorts by latest DATADATE, then FDATE and PDATE, and keeps one observation; after the UPD filter the key and selected-vintage duplicate counts are both zero. No market or accounting table is merged in Stata.

The source's starting population is U.S.-incorporated firm-years outside financial services and utilities with nonmissing assets, in the CRSP/Compustat intersection. I use FIC=USA and historical SIC to exclude 6000–6999 (financial services) and 4900–4999 (utilities); these SIC ranges are the documented operational definition where the article does not give numeric cutoffs. A CRSP firm-year requires a CCM link overlapping the 12 months ending at DATADATE, a primary link marker P or C, a link type LC or LU, and a CRSP daily PERMNO/DATE record in that interval while the link is valid. The CRSP guide defines P/C as primary markers and LC/LU as standard/research-complete or CUSIP-based links. A bounded source probe found 36,658 primary LC/LU links active exactly at DATADATE versus 37,556 firm-years with those links overlapping the fiscal window; I use the latter because the paper calls for a CRSP/Compustat firm-year intersection and does not require that the security remain listed at fiscal year-end. The final daily-date join identifies 37,521 of 49,358 U.S., nonfinancial/nonutility asset firm-years (8,352 firms); it has 37,806 link rows, including 245 firm-years with multiple link rows and 85 with multiple PERMNOs, which are collapsed only to firm-year membership after the daily overlap is checked. This is a best-effort operationalization of the article's unspecific database-intersection rule.

The source lists missing regression variables, a pre-tax reporting loss, negative current tax expense, and a Compustat-identified NOL as final-sample exclusions. PTBI for year t and t+1 is PI less MII when MII is reported, and PI otherwise, following article footnote 18. Current PTBI must be positive. The paper's Table 2 note defines current tax as item #16 less item #50, so the screen uses `TXT - TXDI` where both items are observed; current-vintage `TXC` is compared as a field check but does not replace the stated formula. Only observed negative current-tax amounts are excluded. Missing formula inputs leave tax status unknown and retained, rather than being set to zero or treated as a negative value. This is a best-effort interpretation because the paper does not say whether missing values for this separate sample screen should be dropped. The build reports the number of unknown current-tax classifications and the overlap between TXC and the paper formula. For NOL identification, the current catalog exposes TLCF, whose legacy #52 definition is tax-loss carryforward, rather than a separate NOL indicator. Observed positive TLCF is excluded as the best available NOL screen. Missing TLCF is retained as unclassified rather than changed to zero; the build reports unknown TLCF status. This limitation can leave unobserved NOL firms in the sample.

The article says variables are scaled by average total assets but does not specify the denominator's timing. I use the conventional mean of total assets at t-1 and t for variables at t, and at t and t+1 for PTBI at t+1. Both asset observations must be positive and observed. PTBI amounts are PI less MII when MII is reported, divided by the period-matched average assets. DTE is the sum of federal and foreign deferred tax expense when both are observed, with total deferred taxes used when either component is missing, divided by the 35% statutory rate to estimate temporary BTD dollars, then scaled by average assets at t. State deferred taxes are excluded. Annual LNBTD and LPBTD indicators mark the bottom and top quintiles of that scaled temporary-BTD measure within each year, among the eligible final-sample firm-years, per Equation (2). The ranking uses the un-winsorized scaled BTD; winsorization does not define the groups.

Currency codes are checked for t-1, t, and t+1. Observations with missing or changing `CURCD` across those periods are excluded because no verified conversion is made. Each regression variable is a ratio of same-currency accounting amounts; ratios are unitless, and no absolute dollar regressor, dollar threshold, or price variable enters this scope. A common USD conversion is therefore not required for the regression units. The build reports the final-sample USD/non-USD currency mix and cross-period currency exclusions. No split-adjusted measure is used; CRSP supplies only a linked trading-day membership check.

Following the article's general 1st/99th-percentile reset, pooled 1st and 99th percentile cutoffs are computed over the eligible 1994–2000 firm-years and applied to scaled PTBI_t, PTBI_t+1, and scaled temporary BTD. The paper does not state whether cutoffs are pooled or year-specific; pooled cutoffs are a documented best-effort assumption. Table 3 Panel B is estimated on the resulting full eligible sample with no additional fixed effects or controls. The small-BTD observations have both group indicators equal to zero.

## Construction diagnostics and sample comparison

The source reports a 47,507-firm-year starting population and a 14,106-firm-year final sample (4,048 firms). The local population before CRSP intersection is 49,358 firm-years and after the date-valid link/daily membership check is 37,521 (8,352 firms); that matched entry sample is 9,986 firm-years below the source count. After the Equation (2) construction and currency checks, 32,363 rows remain. Of these, 117 have an unknown current-tax screen status, which is not coded as zero or excluded; 12,177 have negative adjusted PTBI, 665 have negative observed `TXT - TXDI`, and 3,912 have positive TLCF. The loss screen uses the paper's adjusted PTBI definition: raw PI < 0 flags 12,161 observations, adjusted PTBI < 0 flags 12,177, and three exact-zero PTBI rows are retained. The final sample is 15,609 firm-years (4,572 firms), 1,503 firm-years and 524 firms above the source count. There are 5,687 final observations with unknown TLCF status and 29 with unknown current-tax status. Final currency is USD for 15,603 observations and CAD for six; currencies are stable across t-1, t, and t+1. No final GVKEY-year keys are duplicated.

The article reports 22,242 exclusions for missing regression variables, 6,447 pre-tax reporting losses, 1,518 negative current taxes, and 3,194 Compustat-identified NOLs. This build counts 5,147 missing Equation (2) constructions and 11 currency mismatches, then 12,177 negative adjusted PTBI observations, 665 negative formula-based current taxes, and 3,912 observed positive TLCF values. The source does not enumerate every variable behind its missing-data screen; that count may cover the paper's broader set of regressions, while this frozen build requires the variables for Equation (2) plus the chosen asset scaling and currency checks. The current `TXC` field is missing for many records that have source-formula inputs: in the previous build its missingness removed 2,525 eligible rows, while the paper formula is unavailable for only 117 rows after regression and currency construction. Among 69,578 final-update Compustat rows with all three tax fields observed, `TXC` equals `TXT - TXDI` within $0.0001 million for 65,951 (94.79%) and differs for 3,627 (5.21%); differences range from -$848.4 million to $1,357.0 million. The A2 build therefore follows the paper formula, not TXC. The local TLCF field cannot classify 5,687 final observations. These differences do not establish a data-vintage cause, so counts and estimates are recorded without tuning to the published N.

For the winsorized scaled variables, the final sample has current PTBI quartiles 0.0594, 0.1071, and 0.1711, and next-year PTBI quartiles 0.0296, 0.0877, and 0.1530. The corresponding published Table 1 values are 0.066, 0.113, 0.176 and 0.0368, 0.093, 0.157. The local bottom/top annual BTD groups contain 3,120 observations each (19.99% per group), compared with 2,823 each in the source. Group indicators are formed by within-year `PROC RANK GROUPS=5` on un-winsorized scaled BTD before applying the pooled 1st/99th-percentile resets. Ties retain SAS's default mean-rank handling; published group sizes are not used to select observations.

The reproduced regression has N=15,609, R-squared 0.2618, and adjusted R-squared 0.2616. In source order, the intercept is -0.0098 (t=-5.20), LNBTD 0.0176 (t=4.60), LPBTD 0.0175 (t=4.19), PTBI_t 0.7581 (t=58.86), PTBI_t×LNBTD -0.0904 (t=-4.09), and PTBI_t×LPBTD -0.2239 (t=-8.37). The interaction equality test is F=20.41 (p=0.0000063). The published estimates are, in the same order, -0.003 (t=-1.62), 0.017 (t=4.43), 0.017 (t=4.00), 0.742 (t=57.35), -0.100 (t=-4.56), and -0.212 (t=-7.99), with adjusted R-squared 0.272 and equality test F=15.01 (p=0.0001). The two local interaction estimates are negative and individually significant at p<0.001; their difference is also significant. The local first interaction is less negative and the second more negative than in the source; R-squared is lower, and the local intercept is significant although the source intercept is not. These differences are recorded without recalibrating the specification. Export p-values round to 0.000 at three decimal places; Stata logs preserve t-statistics and the equality-test p-value.

## Outputs and remaining uncertainty

`build.sas` produces the one-row-per-GVKEY/fiscal-year SAS analysis dataset and ordinary counts for report-vintage selection, link multiplicity, sample attrition, currency consistency, annual BTD-group counts, and the final firm-year/firm totals. `stata.ipynb` contains a direct OLS estimation cell with the coefficient-equality F-test plus a sample descriptive-statistics cell for PTBI and scaled BTD. P-values in the export are two-tailed, matching the stated Table 3 convention; the source does not identify clustering, robust standard errors, exact average-assets timing, SIC numeric cutoffs, or the NOL flag's current-vintage mapping. These are assumptions/limitations, not fit targets.

The A2 SAS build and native Stata handoff completed successfully. Both assigned Stata cells returned code 0. All four required formats were produced for the Table 3 Panel B regression and the sample descriptive table. The Stata regression log includes coefficients, conventional OLS inference, R-squared, adjusted R-squared, N, and the interaction-equality F-test. No assigned fit is blocked. A1 outputs remain preserved in attempt_01; the A2 current-tax formula and missing-status handling are documented as source-based repairs.


In [ ]:
options errorabend mprint source2;

libname comp "/storage/sas/base/comp/d_na" access=readonly;
libname ccm "/storage/sas/base/crsp/a_ccm" access=readonly;
libname crspst "/storage/sas/base/crsp/a_stock" access=readonly;
libname out "./data";

title "W3121890812 Table 3 Panel B construction diagnostics";

data work.comp_all;
    set comp.funda(
        keep=gvkey datadate fyear fic sich indfmt consol popsrc datafmt
             curcd upd fdate pdate pi mii at txc txt tlcf txdfed txdfo txdi
        where=(fyear >= 1993 and fyear <= 2001 and
               indfmt = 'INDL' and consol = 'C' and popsrc = 'D' and
               datafmt = 'STD')
    );
    if missing(gvkey) or missing(datadate) or missing(fyear) then delete;
    format datadate fdate pdate yymmdd10.;
run;

proc freq data=work.comp_all;
    title "Update-code distribution before final-report filter";
    tables fyear*upd / missing;
run;

proc sql;
    title "Effect of selecting final Compustat updates";
    select count(*) as standard_rows,
           sum(case when upd = 3 then 1 else 0 end) as upd3_rows,
           sum(case when upd ne 3 or missing(upd) then 1 else 0 end) as other_upd_rows
      from work.comp_all;
quit;

data work.comp0;
    set work.comp_all;
    if upd ne 3 then delete;
run;

proc sql;
    title "Compustat final standard annual report keys";
    select count(*) as report_rows from work.comp0;
    select count(*) as duplicate_gvkey_fyear_groups
      from (select gvkey, fyear from work.comp0
            group by gvkey, fyear having count(*) > 1) as d;
    select count(*) as dup_vintage_ties
      from (select gvkey, fyear, datadate, fdate, pdate
              from work.comp0
              group by gvkey, fyear, datadate, fdate, pdate
              having count(*) > 1) as d;
quit;

proc sort data=work.comp0 out=work.comp_order;
    by gvkey fyear descending datadate descending fdate descending pdate;
run;

data work.comp_year;
    set work.comp_order;
    by gvkey fyear;
    if first.fyear;
run;

data work.tax_compare;
    set work.comp_year;
    if nmiss(txc, txt, txdi) = 0;
    current_tax_from_items = txt - txdi;
    tax_diff = txc - current_tax_from_items;
    tax_match = (abs(tax_diff) <= 0.0001);
run;

proc freq data=work.tax_compare;
    title "Direct current-tax field versus paper item formula";
    tables tax_match / missing;
run;

proc means data=work.tax_compare n mean min p25 median p75 max;
    var tax_diff;
run;

data work.t_base;
    set work.comp_year;
    if fyear < 1994 or fyear > 2000 then delete;
    if fic ne 'USA' then delete;
    if missing(sich) or missing(at) then delete;
    if (sich >= 4900 and sich <= 4999) or
       (sich >= 6000 and sich <= 6999) then delete;
    crsp_start = intnx('year', datadate, -1, 'same') + 1;
    crsp_end = datadate;
    format crsp_start crsp_end yymmdd10.;
run;

proc sql;
    title "Source starting population before CRSP intersection";
    select count(*) as init_us_nonfin_n,
           count(distinct gvkey) as companies
      from work.t_base;
quit;

proc sql;
    create table work.link_fy as
    select a.gvkey, a.fyear, a.datadate, a.crsp_start, a.crsp_end,
           b.lpermno, b.linkprim, b.linktype, b.linkdt, b.linkenddt
      from work.t_base as a
      inner join ccm.ccmxpf_linktable(
          keep=gvkey linkprim linktype lpermno linkdt linkenddt
          where=((missing(linkdt) or linkdt <= '31DEC2001'd) and
                 (missing(linkenddt) or linkenddt >= '01JAN1993'd))
      ) as b
        on a.gvkey = b.gvkey
     where b.lpermno > 0 and b.linkprim in ('P', 'C') and
           b.linktype in ('LC', 'LU') and
           (missing(b.linkdt) or b.linkdt <= a.crsp_end) and
           (missing(b.linkenddt) or b.linkenddt >= a.crsp_start);

    create table work.link_perm as
    select distinct lpermno as permno from work.link_fy;

    title "CRSP link multiplicity before daily-date intersection";
    select count(*) as primary_window_link_rows,
           count(distinct lpermno) as distinct_permnos
      from work.link_fy;
    select count(*) as multi_link_years
      from (select gvkey, fyear from work.link_fy
            group by gvkey, fyear having count(*) > 1) as d;
    select count(*) as multi_permno_years
      from (select gvkey, fyear from work.link_fy
            group by gvkey, fyear having count(distinct lpermno) > 1) as d;
quit;

data work.dsf_match(keep=permno date);
    length permno 8;
    if _n_ = 1 then do;
        declare hash h(dataset:"work.link_perm");
        h.defineKey("permno");
        h.defineDone();
    end;
    set crspst.dsf(
        keep=permno date
        where=(date >= '01JAN1993'd and date <= '31DEC2001'd)
    );
    if h.check(key:permno) = 0;
    format date yymmdd10.;
run;

proc sql;
    create table work.crsp_fy as
    select distinct a.gvkey, a.fyear
      from work.link_fy as a
      inner join work.dsf_match as d
        on a.lpermno = d.permno and
           d.date >= a.crsp_start and d.date <= a.crsp_end and
           (missing(a.linkdt) or d.date >= a.linkdt) and
           (missing(a.linkenddt) or d.date <= a.linkenddt);

    title "CRSP daily records and unique firm-year intersection";
    select count(*) as candidate_permno_dates from work.dsf_match;
    select count(*) as crsp_intersection_firmyears from work.crsp_fy;
quit;

proc sort data=work.t_base out=work.t_base_s;
    by gvkey fyear;
run;
proc sort data=work.crsp_fy out=work.crsp_fy_s;
    by gvkey fyear;
run;

data work.base;
    merge work.t_base_s(in=in_base) work.crsp_fy_s(in=in_crsp);
    by gvkey fyear;
    if in_base and in_crsp;
    crsp_ok = 1;
run;

proc sql;
    title "Initial sample after date-valid CRSP/Compustat intersection";
    select count(*) as initial_firmyears,
           count(distinct gvkey) as initial_companies
      from work.base;
quit;

proc sql;
    create table work.panel as
    select t.gvkey, t.fyear, t.datadate, t.sich, t.curcd as cur_t,
           t.pi as pi_t, t.mii as mii_t, t.at as at_t,
           t.txt as txt_t, t.tlcf as tlcf_t,
           t.txdfed as txdfed_t, t.txdfo as txdfo_t, t.txdi as txdi_t,
           p.at as at_tm1, p.curcd as cur_tm1,
           n.pi as pi_tp1, n.mii as mii_tp1, n.at as at_tp1,
           n.curcd as cur_tp1
      from work.base as t
      left join work.comp_year as p
        on t.gvkey = p.gvkey and p.fyear = t.fyear - 1
      left join work.comp_year as n
        on t.gvkey = n.gvkey and n.fyear = t.fyear + 1;
quit;

data work.screen_flags;
    set work.panel;
    ptbi_amt_t = .;
    ptbi_amt_tp1 = .;
    if not missing(pi_t) then do;
        ptbi_amt_t = pi_t;
        if not missing(mii_t) then ptbi_amt_t = pi_t - mii_t;
    end;
    if not missing(pi_tp1) then do;
        ptbi_amt_tp1 = pi_tp1;
        if not missing(mii_tp1) then ptbi_amt_tp1 = pi_tp1 - mii_tp1;
    end;

    avg_at_t = .;
    avg_at_tp1 = .;
    if nmiss(at_tm1, at_t) = 0 then avg_at_t = (at_tm1 + at_t) / 2;
    if nmiss(at_t, at_tp1) = 0 then avg_at_tp1 = (at_t + at_tp1) / 2;
    if avg_at_t <= 0 then avg_at_t = .;
    if avg_at_tp1 <= 0 then avg_at_tp1 = .;

    ptbi_t = .;
    ptbi_tp1 = .;
    if not missing(ptbi_amt_t) and not missing(avg_at_t) then
        ptbi_t = ptbi_amt_t / avg_at_t;
    if not missing(ptbi_amt_tp1) and not missing(avg_at_tp1) then
        ptbi_tp1 = ptbi_amt_tp1 / avg_at_tp1;

    dte_tax = .;
    if nmiss(txdfed_t, txdfo_t) = 0 then dte_tax = txdfed_t + txdfo_t;
    else if not missing(txdi_t) then dte_tax = txdi_t;
    btd_scaled = .;
    if not missing(dte_tax) and not missing(avg_at_t) then
        btd_scaled = (dte_tax / 0.35) / avg_at_t;

    current_tax_t = .;
    current_tax_method = 0;
    if nmiss(txt_t, txdi_t) = 0 then do;
        current_tax_t = txt_t - txdi_t;
        current_tax_method = 1;
    end;
    if missing(current_tax_t) then tax_unknown = 1;
    else tax_unknown = 0;
    if not missing(current_tax_t) and current_tax_t < 0 then negtax_flag = 1;
    else negtax_flag = 0;

    cur_ok = 0;
    if not missing(cur_tm1) and not missing(cur_t) and not missing(cur_tp1) then do;
        if cur_tm1 = cur_t and cur_t = cur_tp1 then cur_ok = 1;
    end;

    miss_core = 0;
    if missing(ptbi_t) or missing(ptbi_tp1) or missing(btd_scaled) or
       missing(avg_at_t) or missing(avg_at_tp1) then miss_core = 1;

    mii_missing_t = missing(mii_t);
    mii_missing_tp1 = missing(mii_tp1);
    mii_zero_t = (not missing(mii_t) and mii_t = 0);
    if not missing(ptbi_t) and ptbi_t < 0 then loss_flag = 1;
    else loss_flag = 0;
    if not missing(pi_t) and pi_t < 0 then raw_pi_loss_flag = 1;
    else raw_pi_loss_flag = 0;
    if not missing(ptbi_amt_t) and ptbi_amt_t = 0 then zero_ptbi_flag = 1;
    else zero_ptbi_flag = 0;
    if missing(tlcf_t) then nol_unknown = 1;
    else nol_unknown = 0;
    if not missing(tlcf_t) and tlcf_t > 0 then nol_flag = 1;
    else nol_flag = 0;
run;

proc sql;
    title "Sequential sample-screen counts";
    select count(*) as crsp_intersection_firmyears,
           sum(case when miss_core = 1 then 1 else 0 end) as missing_regression_construction,
           sum(case when miss_core = 0 and cur_ok = 0 then 1 else 0 end) as currency_mismatch,
           sum(case when miss_core = 0 and cur_ok = 1 and tax_unknown = 1 then 1 else 0 end) as missing_current_tax,
           sum(case when miss_core = 0 and cur_ok = 1 then 1 else 0 end) as after_reg_cur,
           sum(case when miss_core = 0 and cur_ok = 1 and raw_pi_loss_flag = 1 then 1 else 0 end) as raw_pi_losses,
           sum(case when miss_core = 0 and cur_ok = 1 and zero_ptbi_flag = 1 then 1 else 0 end) as zero_ptbi_rows,
           sum(case when miss_core = 0 and cur_ok = 1 and loss_flag = 1 then 1 else 0 end) as current_ptbi_losses,
           sum(case when miss_core = 0 and cur_ok = 1 and loss_flag = 0 and negtax_flag = 1 then 1 else 0 end) as negative_current_tax,
           sum(case when miss_core = 0 and cur_ok = 1 and loss_flag = 0 and negtax_flag = 0 and nol_flag = 1 then 1 else 0 end) as positive_tlcf_nol,
           sum(case when miss_core = 0 and cur_ok = 1 and loss_flag = 0 and negtax_flag = 0 and nol_flag = 0 then 1 else 0 end) as final_firmyears,
           sum(case when miss_core = 0 and cur_ok = 1 and loss_flag = 0 and negtax_flag = 0 and nol_flag = 0 and nol_unknown = 1 then 1 else 0 end) as final_nol_unknown,
           sum(case when miss_core = 0 and cur_ok = 1 and loss_flag = 0 and negtax_flag = 0 and nol_flag = 0 and tax_unknown = 1 then 1 else 0 end) as final_tax_unknown
      from work.screen_flags;
quit;

data work.eligible;
    set work.screen_flags;
    if miss_core ne 0 then delete;
    if cur_ok ne 1 then delete;
    if loss_flag ne 0 then delete;
    if negtax_flag ne 0 then delete;
    if nol_flag ne 0 then delete;
run;

proc freq data=work.eligible;
    title "Annual observations available for Equation (2) BTD ranks";
    tables fyear / missing;
run;

proc sort data=work.eligible out=work.eligible_s;
    by fyear;
run;
proc rank data=work.eligible_s out=work.ranked groups=5;
    by fyear;
    var btd_scaled;
    ranks btd_rank;
run;

data work.ranked;
    set work.ranked;
    lnbtd = (btd_rank = 0);
    lpbtd = (btd_rank = 4);
run;

proc freq data=work.ranked;
    title "Annual bottom/top temporary-BTD quintile group sizes";
    tables fyear*btd_rank fyear*lnbtd fyear*lpbtd / missing;
run;

proc univariate data=work.ranked noprint;
    var ptbi_t ptbi_tp1 btd_scaled;
    output out=work.win_limits
        pctlpts=1 99
        pctlpre=ptbit_ ptbin_ btd_
        pctlname=p1 p99;
run;

proc print data=work.win_limits noobs;
    title "Pooled 1st and 99th percentile reset cutoffs";
run;

data work.final;
    if _n_ = 1 then set work.win_limits;
    set work.ranked;
    ptbi_t_raw = ptbi_t;
    ptbi_tp1_raw = ptbi_tp1;
    btd_raw = btd_scaled;
    ptbi_t = max(ptbit_p1, min(ptbi_t, ptbit_p99));
    ptbi_tp1 = max(ptbin_p1, min(ptbi_tp1, ptbin_p99));
    btd_scaled = max(btd_p1, min(btd_scaled, btd_p99));
    ptbi_lnbtd = ptbi_t * lnbtd;
    ptbi_lpbtd = ptbi_t * lpbtd;
    if missing(tlcf_t) then nol_unknown = 1;
    else nol_unknown = 0;
    mii_missing_t = missing(mii_t);
    mii_missing_tp1 = missing(mii_tp1);
    mii_zero_t = (not missing(mii_t) and mii_t = 0);
    label ptbi_t = "PTBI at t scaled by average assets"
          ptbi_tp1 = "PTBI at t+1 scaled by average assets"
          btd_scaled = "Scaled temporary book-tax difference"
          lnbtd = "Large negative temporary BTD"
          lpbtd = "Large positive temporary BTD"
          ptbi_lnbtd = "PTBI at t x LNBTD"
          ptbi_lpbtd = "PTBI at t x LPBTD"
          cur_t = "Compustat reporting currency at t"
          nol_unknown = "TLCF status missing; not coded as zero";
    label tax_unknown = "Current tax could not be computed from available items"
          current_tax_method = "1=TXT-TXDI, 0=unknown"
          mii_missing_t = "MII at t not reported"
          mii_missing_tp1 = "MII at t+1 not reported"
          mii_zero_t = "MII at t reported as zero";
    keep gvkey fyear datadate sich cur_t ptbi_t ptbi_tp1 btd_scaled
         lnbtd lpbtd ptbi_lnbtd ptbi_lpbtd btd_rank ptbi_t_raw
         ptbi_tp1_raw btd_raw nol_unknown tax_unknown current_tax_method
         mii_missing_t mii_missing_tp1 mii_zero_t;
run;

proc sort data=work.final out=out.book_tax_firm_years;
    by gvkey fyear;
run;

proc sql;
    title "Final SAS handoff checks";
    select count(*) as final_firmyears,
           count(distinct gvkey) as final_companies,
           sum(nol_unknown) as unknown_tlcf_status,
           sum(tax_unknown) as unknown_current_tax_status,
           sum(mii_missing_t) as missing_mii_t,
           sum(mii_zero_t) as observed_zero_mii_t,
           sum(case when cur_t = 'USD' then 1 else 0 end) as usd_firmyears,
           sum(case when cur_t ne 'USD' then 1 else 0 end) as nonusd_firmyears
      from out.book_tax_firm_years;
    select count(*) as dup_final_keys
      from (select gvkey, fyear from out.book_tax_firm_years
            group by gvkey, fyear having count(*) > 1) as d;
quit;

proc means data=out.book_tax_firm_years n nmiss min p1 p25 median p75 p99 max;
    title "Final analysis-variable distribution check";
    var ptbi_t ptbi_tp1 btd_scaled;
run;

proc freq data=out.book_tax_firm_years;
    title "Final analysis groups and currencies";
    tables lnbtd lpbtd cur_t nol_unknown tax_unknown current_tax_method
           mii_missing_t mii_missing_tp1 mii_zero_t / missing;
run;

title;
